# **Tahap 2**


**Ekstraksi Metadata**

In [ ]:
import os
import re
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import re

def extract_metadata(text):
    metadata = {}

    nomor = re.search(r'(putusan|penetapan)?\s*nomor[: ]?([0-9A-Za-z./-]+)', text, re.IGNORECASE)
    metadata['Nomor Perkara'] = nomor.group(2).strip() if nomor else None
    tanggal = re.search(r'tanggal\s+(\d{1,2}\s+[A-Za-z]+\s+\d{4})', text, re.IGNORECASE)
    metadata['Tanggal Putusan'] = tanggal.group(1).strip() if tanggal else None

    if "pengadilan militer" in text.lower():
        metadata['Jenis Perkara'] = "Pidana Militer"
    elif "pdt.p" in text.lower():
        metadata['Jenis Perkara'] = "Perdata (Permohonan)"
    else:
        metadata['Jenis Perkara'] = "Pidana Umum"

    pasal = list(set(re.findall(r'pasal\s+\d+(?:\s+ayat\s*\(\d+\))?(?:\s+ke-\d+)?', text.lower())))
    metadata['Pasal'] = pasal if pasal else None

    penggugat = re.search(r'(pemohon|penggugat|terdakwa|nama lengkap)\s*:\s*([^\n;]+)', text, re.IGNORECASE)
    metadata['Pihak Penggugat/Pemohon'] = penggugat.group(2).strip() if penggugat else None


    terdakwa = re.search(
        r'nama\s*(?:lengkap)?\s*:\s*(.+?)(?=\s*[\.;,]*\s*(?:pangkat|nrp|j\s*a\s*b\s*a\s*t\s*a\s*n|kesatuan|tempat|tgl|lahir))',
        text,
        re.IGNORECASE
    )
    metadata['Pihak Tergugat/Termohon/Terdakwa'] = terdakwa.group(1).strip() if terdakwa else None

    return metadata

folder_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/data_bersih"
all_metadata = []

if os.path.exists(folder_path):
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                text = f.read()
                meta = extract_metadata(text)

                if meta['Jenis Perkara'] == "Pidana Militer":
                    meta['Pihak Penggugat/Pemohon'] = "Oditur Militer"

                meta['File'] = filename
                all_metadata.append(meta)

    print("=== HASIL EKSTRAKSI ===")
    for data in all_metadata:
        print("\n---")
        for k, v in data.items():
            print(f"{k}: {v}")
else:
    print(f"folder {folder_path} tidak ketemu.")

=== HASIL EKSTRAKSI ===

---
Nomor Perkara: 58-k/pm.ii-09/ad/iii/2025
Tanggal Putusan: 16 oktober 2024
Jenis Perkara: Pidana Militer
Pasal: ['pasal 86 ke-1', 'pasal 173 ayat (1)', 'pasal 58', 'pasal 175 ayat (1)', 'pasal 171', 'pasal 44', 'pasal 85', 'pasal 1', 'pasal 172 ayat (1)', 'pasal 176', 'pasal 88 ayat (1) ke-2', 'pasal 190 ayat (1)', 'pasal 46 ayat (1)', 'pasal 88 ayat (1) ke-1']
Pihak Penggugat/Pemohon: Oditur Militer
Pihak Tergugat/Termohon/Terdakwa: boby pranata ginting
File: case_001.txt

---
Nomor Perkara: 26-k/pmt.iii/bdg/ad/iii/2025
Tanggal Putusan: 22 juni 2024
Jenis Perkara: Pidana Militer
Pasal: ['pasal 126', 'pasal 372', 'pasal 26', 'pasal 228 ayat (1)']
Pihak Penggugat/Pemohon: Oditur Militer
Pihak Tergugat/Termohon/Terdakwa: rasid
File: case_002.txt

---
Nomor Perkara: 18-k/pm.ii-11/ad/iv/2025
Tanggal Putusan: 21 februari 2025
Jenis Perkara: Pidana Militer
Pasal: ['pasal 2 ayat (4)', 'pasal 46 ayat (1)', 'pasal 141 ayat (10)', 'pasal 58', 'pasal 87 ayat (1) ke-2',

**Ekstrasi konten kunci**


1.   Ringkasan Fakta



In [6]:
import re
import os

def extract_fakta(text):
    fakta = {}

    text_clean = re.sub(r'\s+', ' ', text)
    text_clean = re.sub(r'halaman putusan nomor [\w\-\./]+\s*', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'salina\s*pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'\s+', ' ', text_clean)

    dakwaan_text = None

    pola_kutip = re.search(
        r'(?:didakwa melakukan tindak pidana|dengan dakwaan|dakwaan|melakukan pelanggaran lalu lintas|bersalah melakukan tindak pidana)\s*[:]?\s*(?:kesatu|pertama|primair|tunggal|subsider)?\s*[:]?\s*["\'“\”]([^"\'“\”]+?)["\'“\”]',
        text_clean, re.IGNORECASE
    )

    pola_amar = re.search(
        r'bersalah melakukan (?:tindak pidana|pelanggaran lalu lintas)\s*[:]?\s*(?:kesatu|pertama|tunggal)?\s*[:]?\s*["\'“\”]?([^"\'“\”\.]+?)["\'“\”]?(?=\s*sebagaimana|\s*melanggar|\s*pasal|\s*2\.\s*memidana|\s*menjatuhkan pidana|\s*memidana|\.\s|$)',
        text_clean, re.IGNORECASE
    )

    pola_titik_dua = re.search(
        r'(?:tindak pidana|pelanggaran lalu lintas)\s*:\s*(?:kesatu|pertama|tunggal)?\s*[:]?\s*["\'“\”]?([^"\'“\”\.]+?)["\'“\”]?(?=\s*sebagaimana|\s*melanggar|\s*pasal|\.\s|\s*menimbang|\s*menjatuhkan|$)',
        text_clean, re.IGNORECASE
    )

    pola_khusus_17 = re.search(
        r'pada dakwaan.*?yaitu\s*[:]?\s*["\'“\”]([^"\'“\”]+)["\'“\”]',
        text_clean, re.IGNORECASE
    )

    if pola_kutip:
        dakwaan_text = pola_kutip.group(1).strip()
    elif pola_amar:
        dakwaan_text = pola_amar.group(1).strip()
    elif pola_titik_dua:
        dakwaan_text = pola_titik_dua.group(1).strip()
    elif pola_khusus_17:
        dakwaan_text = pola_khusus_17.group(1).strip()

    if dakwaan_text:
        dakwaan_text = re.sub(r'(?i)\s*(menimbang|memperhatikan|mengingat|mengadili|dan kedua).*', '', dakwaan_text)
        fakta['Dakwaan'] = dakwaan_text[:150] + "..." if len(dakwaan_text) > 150 else dakwaan_text
    else:
        fakta['Dakwaan'] = "Tindak Pidana/Pelanggaran tidak terdeteksi spesifik"


    barang_bukti = re.search(
        r'barang bukti berupa(?: surat)?\s*[:]?\s*(.+?)(?=\s*d\.\s*memerintahkan|\s*d\.\s*membebani|\s*d\.\s*membebankan|\s*4\.\s*membebankan|membebankan biaya|menetapkan supaya|demikian diputuskan|menimbang|membaca;)',
        text_clean, re.IGNORECASE
    )

    if barang_bukti:
        fakta['Barang Bukti'] = barang_bukti.group(1).strip()
    else:
        fakta['Barang Bukti'] = "Tidak ditemukan"

    return fakta

folder_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/data_bersih"
all_fakta = []

for filename in sorted(os.listdir(folder_path)):
    if filename.endswith(".txt"):
        with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
            text = f.read()
            fakta = extract_fakta(text)
            fakta['File'] = filename
            all_fakta.append(fakta)

print("=== HASIL EKSTRAKSI (VERSI ANTI NGAWUR) ===\n")
for data in all_fakta:
    print("-" * 60)
    print(f"File         : {data['File']}")
    print(f"Dakwaan      : {data['Dakwaan']}")

    bb = data['Barang Bukti']
    bb_tampil = bb[:300] + " ... (dipotong)" if len(bb) > 300 else bb
    print(f"Barang Bukti : {bb_tampil}")

print("-" * 60)
print("=== PROSES SELESAI ===")

=== HASIL EKSTRAKSI (VERSI ANTI NGAWUR) ===

------------------------------------------------------------
File         : case_001.txt
Dakwaan      : militer yang dengan sengaja melakukan ketidak hadiran tanpa izin dalam waktu damai minimal satu hari dan tidak lebih lama dari tiga puluh hari,ketika ...
Barang Bukti : surat-surat : - 2 (dua) lembar daftar absensi anggota baterai tempur budhi yonarmed 13/nanggala/1/1 kostrad bulan september dan oktober yang ditandatangani oleh danraipur budhi yonarmed 13/nanggala/1/1 kostrad a.n. lettu arm anzar al akbar, s.tr.(han) nrp 1118002920 0495, tetap dilekatkan dalam berk ... (dipotong)
------------------------------------------------------------
File         : case_002.txt
Dakwaan      : barang siapa dengan sengaja dan melawan hukum mengaku sebagai milik sendiri barang sesuatu yang seluruhnya atau sebagian adalah kepunyaan orang lain, ...
Barang Bukti : 1) berupa surat: a) 1 (satu) rangkap fotokopi wabku honor sisfopers tw. i 2024 yonif 431/ssp;

In [7]:
import re
import os
from collections import Counter

def extract_fakta(text):
    fakta = {}

    text_clean = re.sub(r'\s+', ' ', text)
    text_clean = re.sub(r'halaman putusan nomor [\w\-\./]+\s*', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'salina\s*pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'\s+', ' ', text_clean)

    dakwaan_text = None
    p_dakwaan = [
        r'(?:didakwa melakukan tindak pidana|dengan dakwaan|dakwaan|melakukan pelanggaran lalu lintas|bersalah melakukan tindak pidana)\s*[:]?\s*(?:kesatu|pertama|primair|tunggal|subsider)?\s*[:]?\s*["\'“\”]([^"\'“\”]+?)["\'“\”]',
        r'bersalah melakukan (?:tindak pidana|pelanggaran lalu lintas)\s*[:]?\s*(?:kesatu|pertama|tunggal)?\s*[:]?\s*["\'“\”]?([^"\'“\”\.]+?)["\'“\”]?(?=\s*sebagaimana|\s*melanggar|\s*pasal|\s*2\.\s*memidana|\.\s|$)',
        r'pada dakwaan.*?yaitu\s*[:]?\s*["\'“\”]([^"\'“\”]+)["\'“\”]'
    ]
    for p in p_dakwaan:
        res = re.search(p, text_clean, re.IGNORECASE)
        if res:
            dakwaan_text = res.group(1).strip()
            break

    if dakwaan_text:
        dakwaan_text = re.sub(r'(?i)\s*(menimbang|memperhatikan|mengingat|mengadili|dan kedua).*', '', dakwaan_text)
        fakta['Dakwaan'] = (dakwaan_text[:150] + "...") if len(dakwaan_text) > 150 else dakwaan_text
    else:
        fakta['Dakwaan'] = "Tidak terdeteksi"

    pola_pasal = re.search(r'(pasal\s+\d+[\w\s\(\)\.]+?(?:kuhp[m]?|undang-undang[^,.\n]+))', text_clean, re.IGNORECASE)
    fakta['Pasal'] = pola_pasal.group(1).strip() if pola_pasal else "Tidak terdeteksi"

    pola_vonis_penjara = re.search(
        r'(penjara|kurungan)\s*(?:selama)?\s*[:]?\s*([^;,\.\n]*?(?:tahun|bulan|hari|seumur hidup))',
        text_clean, re.IGNORECASE
    )

    pola_vonis_denda = re.search(
        r'(?:denda|membayar denda)\s*(?:sebesar|sejumlah)?\s*[:]?\s*(rp\s*[\d\.,]+|\d+\s*(?:rupiah|ribu|juta))',
        text_clean, re.IGNORECASE
    )

    if pola_vonis_penjara:
        jenis = pola_vonis_penjara.group(1).capitalize()
        detail = pola_vonis_penjara.group(2).strip()
        detail = re.sub(r'^:\s*', '', detail)
        fakta['Putusan'] = f"{jenis} {detail}"
    elif pola_vonis_denda:
        fakta['Putusan'] = f"Denda {pola_vonis_denda.group(1).strip()}"
    else:
        fakta['Putusan'] = "Tidak terdeteksi/Bebas"

    barang_bukti = re.search(r'barang bukti berupa(?: surat)?\s*[:]?\s*(.+?)(?=\s*d\.\s*memerintahkan|\s*d\.\s*membebani|\s*4\.\s*membebankan|membebankan biaya|menetapkan supaya|demikian diputuskan|menimbang)', text_clean, re.IGNORECASE)
    fakta['Barang Bukti'] = barang_bukti.group(1).strip() if barang_bukti else "Tidak ditemukan"


    kata_list = text_clean.split()
    fakta['Length'] = len(kata_list)
    stop_words = {'yang', 'dan', 'di', 'dengan', 'dari', 'oleh', 'untuk', 'pada', 'bahwa', 'ini', 'terdakwa', 'pengadilan', 'perkara', 'sebagai', 'dalam', 'menimbang', 'kepada'}
    words_filtered = [word.lower() for word in kata_list if word.lower() not in stop_words and len(word) > 3]
    fakta['BoW'] = dict(Counter(words_filtered).most_common(5))

    fakta['QA_Pairs'] = [
        {"Q": "Apa tindak pidana yang didakwakan?", "A": fakta['Dakwaan']},
        {"Q": "Berapa lama putusan pidana yang dijatuhkan?", "A": fakta['Putusan']}
    ]

    return fakta

folder_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/data_bersih"
all_fakta = []

for filename in sorted(os.listdir(folder_path)):
    if filename.endswith(".txt"):
        with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
            text = f.read()
            fakta = extract_fakta(text)
            fakta['File'] = filename
            all_fakta.append(fakta)

print("=== HASIL FINAL EKSTRAKSI & FEATURE ENGINEERING ===\n")
for data in all_fakta:
    print("-" * 60)
    print(f"File         : {data['File']}")
    print(f"Dakwaan      : {data['Dakwaan']}")
    print(f"Pasal Hukum  : {data['Pasal']}")
    print(f"Vonis/Putusan: {data['Putusan']}")

    bb = data['Barang Bukti']
    print(f"Barang Bukti : {bb[:100] + ' ... (dipotong)' if len(bb) > 100 else bb}")

    print("\n[ Feature Engineering ]")
    print(f"Total Kata   : {data['Length']} kata")
    print(f"Top 5 BoW    : {data['BoW']}")
    print(f"QA Pairs     :")
    for qa in data['QA_Pairs']:
        print(f"  Q: {qa['Q']}\n  A: {qa['A']}")

print("-" * 60)
print(f"=== PROSES SELESAI: {len(all_fakta)} file berhasil dibaca ===")

=== HASIL FINAL EKSTRAKSI & FEATURE ENGINEERING ===

------------------------------------------------------------
File         : case_001.txt
Dakwaan      : militer yang dengan sengaja melakukan ketidak hadiran tanpa izin dalam waktu damai minimal satu hari dan tidak lebih lama dari tiga puluh hari,ketika ...
Pasal Hukum  : pasal 171 undang-undang nomor 31 tahun 1997 tentang peradilan militer
Vonis/Putusan: Penjara 9 (sembilan) bulan
Barang Bukti : surat-surat : - 2 (dua) lembar daftar absensi anggota baterai tempur budhi yonarmed 13/nanggala/1/1  ... (dipotong)

[ Feature Engineering ]
Total Kata   : 10157 kata
Top 5 BoW    : {'tanggal': 110, 'tidak': 109, 'kesatuan': 92, 'pidana': 83, 'melakukan': 83}
QA Pairs     :
  Q: Apa tindak pidana yang didakwakan?
  A: militer yang dengan sengaja melakukan ketidak hadiran tanpa izin dalam waktu damai minimal satu hari dan tidak lebih lama dari tiga puluh hari,ketika ...
  Q: Berapa lama putusan pidana yang dijatuhkan?
  A: Penjara 9 (sembilan

**DATA CSV**

In [8]:
import os
import re
import pandas as pd

def extract_gabungan(text, filename):
    data = {}


    text_clean = re.sub(r'\s+', ' ', text)
    text_clean = re.sub(r'halaman putusan nomor [\w\-\./]+\s*', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'salina\s*pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'pelaksanaan fungsi peradilan.*?telp\s*:\s*021-384\s*3348\s*\(ext\.318\)\s*halaman\s*\d+', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'\s+', ' ', text_clean)


    nomor = re.search(r'(putusan|penetapan)?\s*nomor[: ]?([0-9A-Za-z./-]+)', text_clean, re.IGNORECASE)
    data['Nomor Perkara'] = nomor.group(2).strip() if nomor else "Tidak terdeteksi"

    tanggal = re.search(r'tanggal\s+(\d{1,2}\s+[A-Za-z]+\s+\d{4})', text_clean, re.IGNORECASE)
    data['Tanggal Putusan'] = tanggal.group(1).strip() if tanggal else "Tidak terdeteksi"

    if "pengadilan militer" in text_clean.lower():
        data['Jenis Perkara'] = "Pidana Militer"
        data['Pihak Penggugat/Pemohon'] = "Oditur Militer"
    elif "pdt.p" in text_clean.lower():
        data['Jenis Perkara'] = "Perdata (Permohonan)"
        penggugat = re.search(r'(pemohon|penggugat)\s*:\s*([^\n;]+)', text_clean, re.IGNORECASE)
        data['Pihak Penggugat/Pemohon'] = penggugat.group(2).strip() if penggugat else "Tidak terdeteksi"
    else:
        data['Jenis Perkara'] = "Pidana Umum"
        penggugat = re.search(r'(pemohon|penggugat)\s*:\s*([^\n;]+)', text_clean, re.IGNORECASE)
        data['Pihak Penggugat/Pemohon'] = penggugat.group(2).strip() if penggugat else "Tidak terdeteksi"

    pasal = list(set(re.findall(r'pasal\s+\d+(?:\s+ayat\s*\(\d+\))?(?:\s+ke-\d+)?', text_clean.lower())))
    data['Pasal'] = ", ".join(pasal) if pasal else "Tidak terdeteksi"

    terdakwa = re.search(
        r'nama\s*(?:lengkap)?\s*:\s*(.+?)(?=\s*[\.;,]*\s*(?:pangkat|nrp|j\s*a\s*b\s*a\s*t\s*a\s*n|kesatuan|tempat|tgl|lahir))',
        text_clean,
        re.IGNORECASE
    )
    if terdakwa:
        data['Pihak Tergugat/Termohon/Terdakwa'] = terdakwa.group(1).strip().rstrip('.;, ')
    else:
        data['Pihak Tergugat/Termohon/Terdakwa'] = "Tidak terdeteksi"


    dakwaan_text = None
    pola_kutip = re.search(r'(?:didakwa melakukan tindak pidana|dengan dakwaan|dakwaan|melakukan pelanggaran lalu lintas|bersalah melakukan tindak pidana)\s*[:]?\s*(?:kesatu|pertama|primair|tunggal|subsider)?\s*[:]?\s*["\'“\”]([^"\'“\”]+?)["\'“\”]', text_clean, re.IGNORECASE)
    pola_amar = re.search(r'bersalah melakukan (?:tindak pidana|pelanggaran lalu lintas)\s*[:]?\s*(?:kesatu|pertama|tunggal)?\s*[:]?\s*["\'“\”]?([^"\'“\”\.]+?)["\'“\”]?(?=\s*sebagaimana|\s*melanggar|\s*pasal|\s*2\.\s*memidana|\s*menjatuhkan pidana|\s*memidana|\.\s|$)', text_clean, re.IGNORECASE)
    pola_titik_dua = re.search(r'(?:tindak pidana|pelanggaran lalu lintas)\s*:\s*(?:kesatu|pertama|tunggal)?\s*[:]?\s*["\'“\”]?([^"\'“\”\.]+?)["\'“\”]?(?=\s*sebagaimana|\s*melanggar|\s*pasal|\.\s|\s*menimbang|\s*menjatuhkan|$)', text_clean, re.IGNORECASE)
    pola_khusus_17 = re.search(r'pada dakwaan.*?yaitu\s*[:]?\s*["\'“\”]([^"\'“\”]+)["\'“\”]', text_clean, re.IGNORECASE)

    if pola_kutip: dakwaan_text = pola_kutip.group(1).strip()
    elif pola_amar: dakwaan_text = pola_amar.group(1).strip()
    elif pola_titik_dua: dakwaan_text = pola_titik_dua.group(1).strip()
    elif pola_khusus_17: dakwaan_text = pola_khusus_17.group(1).strip()

    if dakwaan_text:
        dakwaan_text = re.sub(r'(?i)\s*(menimbang|memperhatikan|mengingat|mengadili|dan kedua).*', '', dakwaan_text)
        data['Dakwaan'] = dakwaan_text[:150] + "..." if len(dakwaan_text) > 150 else dakwaan_text
    else:
        data['Dakwaan'] = "Tidak terdeteksi"

    barang_bukti = re.search(r'barang bukti berupa(?: surat)?\s*[:]?\s*(.+?)(?=\s*d\.\s*memerintahkan|\s*d\.\s*membebani|\s*d\.\s*membebankan|\s*4\.\s*membebankan|membebankan biaya|menetapkan supaya|demikian diputuskan|menimbang|membaca;)', text_clean, re.IGNORECASE)
    if barang_bukti:
        bb = barang_bukti.group(1).strip()
        data['Barang Bukti'] = bb[:150] + "..." if len(bb) > 150 else bb
    else:
        data['Barang Bukti'] = "Tidak ditemukan"

    return data

folder_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/data_bersih"
output_csv_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/hasil_ekstraksi_putusan.csv"

all_data = []

if not os.path.exists(folder_path):
    print(f"Error: Folder {folder_path} tidak ditemukan. Cek lagi Mount Drive lu ya!")
else:
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                text = f.read()
                hasil = extract_gabungan(text, filename)
                all_data.append(hasil)

    if all_data:
        df = pd.DataFrame(all_data)
        urutan_kolom = [
            'Nomor Perkara',
            'Tanggal Putusan',
            'Jenis Perkara',
            'Pasal',
            'Pihak Penggugat/Pemohon',
            'Pihak Tergugat/Termohon/Terdakwa',
            'Dakwaan',
            'Barang Bukti'
        ]
        df = df[urutan_kolom]

        from google.colab import data_table

        print(f"Berhasil mengekstrak {len(df)} data putusan:")
        display(data_table.DataTable(df, include_index=False, num_rows_per_page=31))



Berhasil mengekstrak 31 data putusan:


,Nomor Perkara,Tanggal Putusan,Jenis Perkara,Pasal,Pihak Penggugat/Pemohon,Pihak Tergugat/Termohon/Terdakwa,Dakwaan,Barang Bukti
0,58-k/pm.ii-09/ad/iii/2025,16 oktober 2024,Pidana Militer,"pasal 86 ke-1, pasal 173 ayat (1), pasal 58, p...",Oditur Militer,boby pranata ginting,militer yang dengan sengaja melakukan ketidak ...,surat-surat : - 2 (dua) lembar daftar absensi ...
1,26-k/pmt.iii/bdg/ad/iii/2025,22 juni 2024,Pidana Militer,"pasal 228 ayat (1), pasal 229, pasal 126, pasa...",Oditur Militer,rasid,barang siapa dengan sengaja dan melawan hukum ...,1) berupa surat: a) 1 (satu) rangkap fotokopi ...
2,18-k/pm.ii-11/ad/iv/2025,21 februari 2025,Pidana Militer,"pasal 2 ayat (4), pasal 46 ayat (1), pasal 141...",Oditur Militer,achmad sarif,desersi dalam waktu damai,yaitu: - 3 (tiga) lembar daftar absensi person...
3,61-k/pm.iii-12/ad/v/2025,24 januari 2025,Pidana Militer,"pasal 87 ayat (1) ke-2, pasal 26, pasal 190 ay...",Oditur Militer,mochammad amin,desersi dalam waktu damai,yaitu: - 6 (enam) lembar daftar absensi korami...
4,62-k/pm.ii-09/au/lv/2025,12 februari 2025,Pidana Militer,"pasal 141 ayat (1), pasal 45, pasal 124 ayat (...",Oditur Militer,wildan mulyana,militer yang dengan sengaja melakukan ketidakh...,1) barang: nihil. 2) surat- surat: a) 2 (dua) ...
5,32-k/pm,14 januari 2025,Pidana Militer,"pasal 2 ayat (4), pasal 46 ayat (1), pasal 26 ...",Oditur Militer,arya prayogi,desersi dimasa damai,-surat: 1) 3 (tiga) lembar daftar absensi pers...
6,36-k/pm.i-02/au/iv/2025,21 januari 2025,Pidana Militer,"pasal 2 ayat (4), pasal 46 ayat (1), pasal 141...",Oditur Militer,herdion saputro,desersi dimasa damai,-surat: 6 (enam) lembar daftar absensi persone...
7,4-p/pm.ii-09/ad/v/2025,22 mei 2025,Pidana Militer,"pasal 211 ayat (4), pasal 288 ayat (1)",Oditur Militer,asep supriadin,mengemudikan kendaraan bermotor di jalan raya ...,- 1 ( satu) buah sim c tni a.n. serka asep sup...
8,363,16 desember 2015,Pidana Militer,"pasal 9 ayat (1), pasal 2 ayat (1), pasal 1, p...",Oditur Militer,"teddy hernayadi, s.e., m.m",korupsi yang dilakukan secara bersama-sama dan...,a. surat-surat : 1) 1 (satu) buku laporan hasi...
9,60-k/pm,18 november 2024,Pidana Militer,"pasal 4 ayat (1), pasal 173 ayat (1), pasal 58...",Oditur Militer,saepul bahri,militer yang karena salahnya atau dengan senga...,-surat : - 1 (satu) lembar daftar absensi a.n....
